In [4]:
import os, glob, random
import numpy as np
import pandas as pd
import torch, timm
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Same seed as notebook 02 -> identical validation split
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Locate the competition data automatically (works regardless of the exact input path)
sub_paths = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
print("sample_submission found at:", sub_paths)
BASE = os.path.dirname(sub_paths[0])
TRAIN_DIR = os.path.join(BASE, "train", "train")
TEST_DIR  = os.path.join(BASE, "test", "test")

meta_path = glob.glob("/kaggle/input/**/train_metadata_clean.csv", recursive=True)
ckpt_paths = sorted(glob.glob("/kaggle/input/**/best_convnext_t_*.pth", recursive=True))
print("Metadata:", meta_path)
print("Checkpoints:", ckpt_paths)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Rebuild labels and the SAME stratified split as notebook 02
df = pd.read_csv(meta_path[0])
CLASSES = sorted(df["class"].unique())
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
df["label"] = df["class"].map(class_to_idx)

_, val_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
val_df = val_df.reset_index(drop=True)
test_df = pd.read_csv(os.path.join(BASE, "sample_submission.csv"))[["image_id"]]

print(f"Val: {len(val_df)} | Test: {len(test_df)}")
print("First 5 val IDs:", val_df["image_id"].head().tolist())

sample_submission found at: ['/kaggle/input/datasets/khlafalshammari/competition/sample_submission.csv']
Metadata: ['/kaggle/input/notebooks/khlafalshammari/01-eda-facial-features/train_metadata_clean.csv']
Checkpoints: ['/kaggle/input/datasets/khlafalshammari/eng707-models/best_convnext_t_128.pth', '/kaggle/input/datasets/khlafalshammari/eng707-models/best_convnext_t_224.pth']
Device: cuda
Val: 9547 | Test: 29443
First 5 val IDs: [39492, 60414, 28776, 10979, 7158]


In [6]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_eval_tf(size):
    # Same deterministic preprocessing as validation/test in notebook 02
    return T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])

class FacialFeatureDataset(Dataset):
    def __init__(self, df, img_dir, transform, has_labels=True):
        self.ids = df["image_id"].values
        self.labels = df["label"].values if has_labels else None
        self.img_dir, self.transform = img_dir, transform
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        img_id = int(self.ids[idx])
        img = Image.open(os.path.join(self.img_dir, f"{img_id:05d}.jpg")).convert("RGB")
        img = self.transform(img)
        return (img, int(self.labels[idx])) if self.labels is not None else (img, img_id)

def make_loader(df, img_dir, size, has_labels):
    ds = FacialFeatureDataset(df, img_dir, make_eval_tf(size), has_labels)
    return DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

In [7]:
import torch.nn.functional as F
from sklearn.metrics import log_loss

@torch.no_grad()
def predict_probs(model, loader):
    """Returns class probabilities for the original images AND their horizontal flips (TTA)."""
    model.eval()
    p_orig, p_flip = [], []
    for x, _ in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            logits_flip = model(torch.flip(x, dims=[3]))   # flip left-right
        p_orig.append(F.softmax(logits.float(), dim=1).cpu())
        p_flip.append(F.softmax(logits_flip.float(), dim=1).cpu())
    return torch.cat(p_orig).numpy(), torch.cat(p_flip).numpy()

val_probs, test_probs = {}, {}
for path in ckpt_paths:
    size = int(os.path.basename(path).split("_")[-1].split(".")[0])   # 128 or 224 from the filename
    model = timm.create_model("convnext_tiny", pretrained=False, num_classes=len(CLASSES))
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    print(f"Model {size}px: predicting validation and test...")
    val_probs[size]  = predict_probs(model, make_loader(val_df,  TRAIN_DIR, size, True))
    test_probs[size] = predict_probs(model, make_loader(test_df, TEST_DIR,  size, False))
    np.save(f"/kaggle/working/test_probs_{size}.npy", np.stack(test_probs[size]))   # reuse in later ensembles
    del model; torch.cuda.empty_cache()

# Compare all combinations on validation
y_val = val_df["label"].values
candidates = {
    "128 (no TTA)":           val_probs[128][0],
    "128 + TTA":              (val_probs[128][0] + val_probs[128][1]) / 2,
    "224 (no TTA)":           val_probs[224][0],
    "224 + TTA":              (val_probs[224][0] + val_probs[224][1]) / 2,
    "Ensemble (no TTA)":      (val_probs[128][0] + val_probs[224][0]) / 2,
    "Ensemble + TTA":         (sum(val_probs[128]) + sum(val_probs[224])) / 4,
}
rows = []
for name, p in candidates.items():
    preds = p.argmax(1)
    rows.append({"setup": name,
                 "val_acc": (preds == y_val).mean(),
                 "errors": int((preds != y_val).sum()),
                 "val_logloss": log_loss(y_val, p, labels=range(len(CLASSES)))})
comparison = pd.DataFrame(rows).set_index("setup").round(5)
comparison.to_csv("/kaggle/working/ensemble_comparison.csv")
comparison

Model 128px: predicting validation and test...


  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/231 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78397580ac00>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78397580ac00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1655, in _shutdown_workers
    self._pin_memory_thread.join()
  File "/usr/lib/python3.13/threading.py", line 1088, in join

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in:     raise RuntimeError("cannot join current thread")
RuntimeError: cannot join current thread
    <function _MultiProcessingDataLoaderIter.__del__ at 0x78397580ac00>self._shutdown_workers()
Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/uti

Model 224px: predicting validation and test...


  0%|          | 0/75 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78397580ac00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78397580ac00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

  0%|          | 0/231 [00:00<?, ?it/s]

,val_acc,errors,val_logloss
setup,,,
128 (no TTA),0.99707,28,0.10028
128 + TTA,0.99675,31,0.10004
224 (no TTA),0.99644,34,0.10267
224 + TTA,0.99644,34,0.10163
Ensemble (no TTA),0.99707,28,0.09980
Ensemble + TTA,0.99717,27,0.09982


In [8]:
def save_submission(probs, name):
    sub = pd.read_csv(os.path.join(BASE, "sample_submission.csv"))
    sub["class"] = [idx_to_class[i] for i in probs.argmax(1)]
    assert len(sub) == 29443 and sub["image_id"].is_unique
    assert set(sub["class"]).issubset(set(CLASSES))
    path = f"/kaggle/working/submission_{name}.csv"
    sub.to_csv(path, index=False)
    print(f"Saved {path}")
    print(sub["class"].value_counts().to_dict(), "\n")

ens_tta_test = (sum(test_probs[128]) + sum(test_probs[224])) / 4
tta_128_test = (test_probs[128][0] + test_probs[128][1]) / 2

save_submission(ens_tta_test, "ensemble_128_224_tta")
save_submission(tta_128_test, "convnext_t_128_tta")

Saved /kaggle/working/submission_ensemble_128_224_tta.csv
{'CatEye': 6303, 'DogEye': 5580, 'CatMouth': 3272, 'CatFace': 3114, 'CatNose': 2872, 'DogFace': 2807, 'DogMouth': 2806, 'DogNose': 2689} 

Saved /kaggle/working/submission_convnext_t_128_tta.csv
{'CatEye': 6311, 'DogEye': 5561, 'CatMouth': 3283, 'CatFace': 3128, 'CatNose': 2866, 'DogFace': 2811, 'DogMouth': 2801, 'DogNose': 2682} 

